# Part 15: Long Context — Why Models Break Past Their Training Length, and How to Fix It

A model advertised with a "128k context window" was almost certainly **not trained** on
128k-token sequences. Training at that length is prohibitively expensive, so labs train
short and then *extend* — and the extension techniques are the subject of this notebook.

We will start by breaking a model. Train it on short sequences, evaluate it on longer ones,
and watch perplexity explode. Then implement, from scratch, the four position-encoding
fixes that made long context practical:

1. **Position Interpolation** — squeeze new positions into the trained range
2. **NTK-aware scaling** — squeeze the low frequencies, spare the high ones
3. **YaRN** — interpolate per frequency band, plus attention temperature
4. **ALiBi** — skip rotation entirely, bias by distance

Then the architectural approaches — **sliding-window attention** and the surprising
**attention sink** phenomenon — and finally the thing that actually limits context in
production: not the math, but **KV cache memory**.

In [2]:
import math
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, '..')
from src.modern import RMSNorm, SwiGLU
from src.train import CharTokenizer

torch.manual_seed(0)
device = 'cpu'

with open('../data/sample_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

tokenizer = CharTokenizer(text)
data = torch.tensor(tokenizer.encode(text), dtype=torch.long)

split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

TRAIN_LEN = 128   # the model will only ever see sequences this long

print(f"Corpus: {len(data):,} tokens, vocab {tokenizer.vocab_size}")
print(f"Train: {len(train_data):,}  Val: {len(val_data):,}")
print(f"Training context length: {TRAIN_LEN}")

Corpus: 11,507 tokens, vocab 62
Train: 10,356  Val: 1,151
Training context length: 128


## 1. A model with a pluggable position scheme

To compare position encodings we need one model whose position handling can be swapped
*after* training. That mirrors reality: RoPE scaling is applied post-hoc to an
already-trained model, sometimes followed by brief fine-tuning.

The design: attention asks a **position scheme** object for either a rotation table
(`cos`, `sin`) or an additive score bias. Everything else — RMSNorm, SwiGLU, the residual
structure — is imported from `src/modern.py`.

In [3]:
class RoPEScheme:
    """
    Standard rotary position embeddings, with a hook for rescaling.

    angle(pos, i) = pos * inv_freq[i],  inv_freq[i] = base^(-2i/head_dim)

    Subclasses override `build_inv_freq` and/or `scale_positions` to implement
    the context-extension methods in section 3.
    """

    name = "RoPE"
    attention_scale = 1.0   # YaRN modifies this

    def __init__(self, head_dim, base=10000.0, train_len=None, target_len=None):
        self.head_dim = head_dim
        self.base = base
        self.train_len = train_len
        self.target_len = target_len

    @property
    def scale_factor(self):
        """How far beyond the training length we are asking the model to go."""
        if not self.train_len or not self.target_len:
            return 1.0
        return max(1.0, self.target_len / self.train_len)

    def build_inv_freq(self):
        return 1.0 / (
            self.base
            ** (torch.arange(0, self.head_dim, 2, dtype=torch.float) / self.head_dim)
        )

    def scale_positions(self, positions):
        return positions

    def tables(self, seq_len):
        """Returns (cos, sin), each (seq_len, head_dim/2)."""
        positions = self.scale_positions(torch.arange(seq_len, dtype=torch.float))
        angles = torch.outer(positions, self.build_inv_freq())
        return angles.cos(), angles.sin()

    def bias(self, seq_len):
        """Additive attention bias. Only ALiBi uses this."""
        return None


def apply_rotation(x, cos, sin):
    """
    Rotate x by the given angles. x: (B, h, T, head_dim).

    Same rotate-half convention as src/modern.py.
    """
    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


print("RoPEScheme defined.")

RoPEScheme defined.


In [4]:
class ToyAttention(nn.Module):
    """Causal multi-head attention with a swappable position scheme."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, scheme, window=None, num_sink=0):
        """
        Args:
            x: (B, T, d_model)
            scheme: a position scheme (RoPEScheme or subclass, or NoPE/ALiBi)
            window: If set, each query attends only to the last `window` keys
            num_sink: With a window, always keep the first `num_sink` positions
                visible (attention sinks, section 4)
        """
        B, T, d = x.shape
        qkv = self.W_qkv(x).view(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)   # each (B, h, T, head_dim)

        cos_sin = scheme.tables(T)
        if cos_sin is not None:
            cos, sin = cos_sin
            q = apply_rotation(q, cos, sin)
            k = apply_rotation(k, cos, sin)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # YaRN raises attention temperature to compensate for interpolation
        scores = scores * scheme.attention_scale

        pos_bias = scheme.bias(T)
        if pos_bias is not None:
            scores = scores + pos_bias

        allowed = torch.ones(T, T, dtype=torch.bool).tril()
        if window is not None:
            # Drop keys older than `window` steps
            too_old = torch.ones(T, T, dtype=torch.bool).tril(diagonal=-window)
            allowed = allowed & ~too_old
            if num_sink > 0:
                # ...but always keep the first num_sink columns
                sink = torch.zeros(T, T, dtype=torch.bool)
                sink[:, :num_sink] = True
                allowed = allowed | (sink & torch.ones(T, T, dtype=torch.bool).tril())

        scores = scores.masked_fill(~allowed, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        out = out.transpose(1, 2).contiguous().view(B, T, d)
        return self.W_o(out)


class ToyBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = ToyAttention(d_model, num_heads)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, scheme, window=None, num_sink=0):
        x = x + self.attn(self.attn_norm(x), scheme, window, num_sink)
        return x + self.ffn(self.ffn_norm(x))


class ToyLM(nn.Module):
    """A small decoder-only LM whose position scheme is supplied at call time."""

    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=4):
        super().__init__()
        self.head_dim = d_model // num_heads
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList(
            [ToyBlock(d_model, num_heads) for _ in range(num_layers)]
        )
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, ids, scheme, targets=None, window=None, num_sink=0):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, scheme, window, num_sink)
        logits = self.head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
            )
        return logits, loss


model = ToyLM(tokenizer.vocab_size).to(device)
print(f"ToyLM: {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"head_dim={model.head_dim}")

ToyLM: 811,904 parameters, head_dim=32


### Training at length 128

In [5]:
def get_batch(source, batch_size, seq_len):
    idx = torch.randint(0, len(source) - seq_len - 1, (batch_size,))
    x = torch.stack([source[i:i + seq_len] for i in idx])
    y = torch.stack([source[i + 1:i + seq_len + 1] for i in idx])
    return x, y


base_scheme = RoPEScheme(model.head_dim, base=10000.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
STEPS = 700
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, total_steps=STEPS, pct_start=0.1
)

model.train()
start = time.time()
losses = []
for step in range(STEPS):
    x, y = get_batch(train_data, 24, TRAIN_LEN)
    _, loss = model(x, base_scheme, targets=y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    losses.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f"  step {step+1:4d}  loss {sum(losses[-50:])/50:.4f}")

print(f"\nTrained in {time.time()-start:.1f}s")
model.eval()

  step  100  loss 1.9596


  step  200  loss 0.6004


  step  300  loss 0.2144


  step  400  loss 0.1136


  step  500  loss 0.0880


  step  600  loss 0.0769


  step  700  loss 0.0734

Trained in 76.4s


ToyLM(
  (embed): Embedding(62, 128)
  (blocks): ModuleList(
    (0-3): 4 x ToyBlock(
      (attn_norm): RMSNorm()
      (attn): ToyAttention(
        (W_qkv): Linear(in_features=128, out_features=384, bias=False)
        (W_o): Linear(in_features=128, out_features=128, bias=False)
      )
      (ffn_norm): RMSNorm()
      (ffn): SwiGLU(
        (w_gate): Linear(in_features=128, out_features=352, bias=False)
        (w_up): Linear(in_features=128, out_features=352, bias=False)
        (w_down): Linear(in_features=352, out_features=128, bias=False)
        (dropout): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (norm): RMSNorm()
  (head): Linear(in_features=128, out_features=62, bias=False)
)

## 2. Breaking it

Now the experiment. Evaluate perplexity at increasing context lengths, using the *same
model* and the *same unmodified RoPE*.

What *should* happen if position encoding generalized: more context means more information
about what comes next, so perplexity should **improve** with length.

What actually happens:

In [6]:
@torch.no_grad()
def perplexity(model, scheme, seq_len, source=None, window=None, num_sink=0,
               num_batches=8, batch_size=4):
    """Mean cross-entropy perplexity over random windows of length seq_len."""
    source = val_data if source is None else source
    if len(source) <= seq_len + 1:
        return float('nan')
    total = 0.0
    for _ in range(num_batches):
        x, y = get_batch(source, batch_size, seq_len)
        _, loss = model(x, scheme, targets=y, window=window, num_sink=num_sink)
        total += loss.item()
    return math.exp(total / num_batches)


eval_lengths = [128, 192, 256, 384, 512]

print(f"{'context':>9} {'perplexity':>12}   (trained at 128)")
print("-" * 40)
baseline_ppl = {}
for L in eval_lengths:
    scheme = RoPEScheme(model.head_dim, base=10000.0)
    ppl = perplexity(model, scheme, L)
    baseline_ppl[L] = ppl
    marker = "  <- trained here" if L == TRAIN_LEN else ""
    print(f"{L:>9} {ppl:>12.2f}{marker}")

blowup = baseline_ppl[512] / baseline_ppl[128]
print(f"\nPerplexity got {blowup:.1f}x WORSE with 4x more context.")
print("More information made the model worse. That is the extrapolation failure.")

  context   perplexity   (trained at 128)
----------------------------------------
      128        94.95  <- trained here


      192        99.55
      256       106.87


      384       131.06


      512       147.44

Perplexity got 1.6x WORSE with 4x more context.
More information made the model worse. That is the extrapolation failure.


### Why it fails

The rotation angle is `pos * inv_freq[i]`, and `pos` is unbounded. At position 400 the
low-frequency dimensions have rotated through angles the model **never saw during
training** — the dot products they produce are simply out of distribution.

Look at the angle each frequency band reaches. The trained region is `pos <= 128`;
everything to the right is territory the model has no experience of.

In [7]:
head_dim = model.head_dim
inv_freq = base_scheme.build_inv_freq()
positions = torch.arange(600, dtype=torch.float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: cumulative angle for a fast, medium and slow frequency band
for idx, style in zip([0, head_dim // 4 - 1, head_dim // 2 - 1],
                      ['-', '--', ':']):
    wavelength = 2 * math.pi / inv_freq[idx].item()
    axes[0].plot(positions, positions * inv_freq[idx], style,
                 label=f'dim pair {idx} (wavelength {wavelength:.0f})')
axes[0].axvline(TRAIN_LEN, color='red', lw=2, label='training length')
axes[0].set_xlabel('position')
axes[0].set_ylabel('cumulative rotation angle (radians)')
axes[0].set_title('Past the training length, angles are unseen')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Right: how many full rotations each band completes within the training window
rotations = TRAIN_LEN * inv_freq / (2 * math.pi)
axes[1].semilogy(range(len(rotations)), rotations, 'o-', color='#2E86AB')
axes[1].axhline(1.0, ls='--', color='red',
                label='one full rotation in training')
axes[1].set_xlabel('dimension pair index (fast -> slow)')
axes[1].set_ylabel('rotations completed within 128 tokens')
axes[1].set_title('High frequencies are well-trained; low ones are not')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

n_undertrained = int((rotations < 1.0).sum())
print(f"{n_undertrained} of {len(rotations)} frequency bands complete less than one")
print(f"full rotation within {TRAIN_LEN} tokens. Those bands have only ever been")
print("observed over a narrow arc -- so they are the ones that break first, and")
print("the ones every scaling method targets.")

10 of 16 frequency bands complete less than one
full rotation within 128 tokens. Those bands have only ever been
observed over a narrow arc -- so they are the ones that break first, and
the ones every scaling method targets.


## 3. Four ways to fix it

### 3.1 Position Interpolation (PI)

The simplest idea: if the model handles positions 0–128, and we want 512, then **divide
every position by 4**. Position 512 is presented as position 128 — inside the trained
range.

`angle = (pos / s) * inv_freq`, where `s = target_len / train_len`.

It works, and it has an obvious cost: adjacent tokens are now only `1/s` as far apart in
angle, so the model's ability to resolve *local* order degrades. We compressed the whole
spectrum, including the high frequencies that were working fine.

In [8]:
class PositionInterpolation(RoPEScheme):
    """Linearly compress positions into the trained range (Chen et al. 2023)."""

    name = "Position Interpolation"

    def scale_positions(self, positions):
        return positions / self.scale_factor


print(f"{'context':>9} {'base RoPE':>12} {'PI':>12}")
print("-" * 36)
pi_ppl = {}
for L in eval_lengths:
    scheme = PositionInterpolation(
        head_dim, train_len=TRAIN_LEN, target_len=L
    )
    pi_ppl[L] = perplexity(model, scheme, L)
    print(f"{L:>9} {baseline_ppl[L]:>12.2f} {pi_ppl[L]:>12.2f}")

  context    base RoPE           PI
------------------------------------
      128        94.95        83.97


      192        99.55       203.98


      256       106.87       385.98


      384       131.06       537.12


      512       147.44       618.97


### 3.2 NTK-aware scaling

The insight that makes PI look crude: **not all frequencies need the same treatment.**

High-frequency bands complete many rotations inside the training window, so the model has
seen their full range — they extrapolate fine and should be left alone. Low-frequency bands
never completed even one rotation, so they are the ones that break.

NTK-aware scaling achieves that by changing the **base** rather than the positions:

`base' = base * s^(d / (d - 2))`

Because `inv_freq[i] = base^(-2i/d)`, raising the base barely affects `i = 0` (the fastest
band) and strongly affects large `i` (the slowest). We stretch exactly where stretching is
needed and leave local resolution intact.

In [9]:
class NTKAwareScaling(RoPEScheme):
    """
    Scale the RoPE base instead of the positions (bloc97, 2023).

    Raising the base slows every band, but the effect grows with band index --
    so slow bands get stretched and fast bands are nearly untouched.
    """

    name = "NTK-aware"

    def build_inv_freq(self):
        s = self.scale_factor
        d = self.head_dim
        adjusted_base = self.base * (s ** (d / (d - 2)))
        return 1.0 / (
            adjusted_base ** (torch.arange(0, d, 2, dtype=torch.float) / d)
        )


# Show that it stretches selectively
plain = RoPEScheme(head_dim).build_inv_freq()
ntk = NTKAwareScaling(head_dim, train_len=TRAIN_LEN, target_len=512).build_inv_freq()
pi_equivalent = plain / 4.0   # PI is equivalent to dividing every frequency by s

print("Effect on each frequency band (target 512, s=4):")
print(f"{'band':>6} {'plain':>12} {'PI (/4)':>12} {'NTK-aware':>12} {'NTK/plain':>11}")
print("-" * 58)
for i in [0, 4, 8, 12, 15]:
    print(f"{i:>6} {plain[i]:>12.6f} {pi_equivalent[i]:>12.6f} "
          f"{ntk[i]:>12.6f} {ntk[i]/plain[i]:>11.3f}")

print("\nPI divides every band by 4. NTK leaves the fastest band at ~0.96 of its")
print("original speed while slowing the slowest to ~1/4 -- precisely targeted.")

Effect on each frequency band (target 512, s=4):
  band        plain      PI (/4)    NTK-aware   NTK/plain
----------------------------------------------------------
     0     1.000000     0.250000     1.000000       1.000
     4     0.100000     0.025000     0.069096       0.691
     8     0.010000     0.002500     0.004774       0.477
    12     0.001000     0.000250     0.000330       0.330
    15     0.000178     0.000044     0.000044       0.250

PI divides every band by 4. NTK leaves the fastest band at ~0.96 of its
original speed while slowing the slowest to ~1/4 -- precisely targeted.


In [10]:
print(f"{'context':>9} {'base':>10} {'PI':>10} {'NTK':>10}")
print("-" * 42)
ntk_ppl = {}
for L in eval_lengths:
    scheme = NTKAwareScaling(head_dim, train_len=TRAIN_LEN, target_len=L)
    ntk_ppl[L] = perplexity(model, scheme, L)
    print(f"{L:>9} {baseline_ppl[L]:>10.2f} {pi_ppl[L]:>10.2f} {ntk_ppl[L]:>10.2f}")

  context       base         PI        NTK
------------------------------------------
      128      94.95      83.97      90.84


      192      99.55     203.98     113.98
      256     106.87     385.98     114.01


      384     131.06     537.12     139.36


      512     147.44     618.97     136.33


### 3.3 YaRN

YaRN ("Yet another RoPE extensioN") makes the per-band idea explicit instead of implicit,
and adds a second correction.

**Per-band interpolation.** For band `i`, compute its wavelength `λ_i = 2π / inv_freq[i]`
and how many times it wraps inside the training window: `r_i = train_len / λ_i`.

- `r_i > β` (default 32): band wrapped many times, fully observed → **do not interpolate**
- `r_i < α` (default 1): band never completed a rotation → **interpolate fully** (PI)
- in between: **linear ramp** between the two

**Attention temperature.** Interpolation packs tokens closer together in angle, which makes
attention distributions flatter and more diffuse. YaRN compensates by scaling attention
logits by `1/t` with `t = 0.1·ln(s) + 1`, sharpening them back. This is a small correction
with a measurable effect, and it is the part most reimplementations forget.

In [11]:
class YaRN(RoPEScheme):
    """
    Per-band interpolation with attention temperature (Peng et al. 2023).

    Bands that were fully exercised during training are left alone; bands that
    never completed a rotation are interpolated; the middle is ramped.
    """

    name = "YaRN"

    def __init__(self, head_dim, base=10000.0, train_len=None, target_len=None,
                 alpha=1.0, beta=32.0):
        super().__init__(head_dim, base, train_len, target_len)
        self.alpha = alpha
        self.beta = beta
        s = self.scale_factor
        # Temperature correction: sharpen attention to offset the flattening
        self.attention_scale = 1.0 / (0.1 * math.log(s) + 1.0) if s > 1 else 1.0

    def build_inv_freq(self):
        s = self.scale_factor
        plain = super().build_inv_freq()
        if s <= 1.0:
            return plain

        # Rotations each band completes inside the training window
        wavelength = 2 * math.pi / plain
        rotations = self.train_len / wavelength

        # ramp = 1 -> keep original (well-trained band)
        # ramp = 0 -> full interpolation (under-trained band)
        ramp = ((rotations - self.alpha) / (self.beta - self.alpha)).clamp(0.0, 1.0)

        interpolated = plain / s
        return interpolated * (1 - ramp) + plain * ramp


yarn = YaRN(head_dim, train_len=TRAIN_LEN, target_len=512)
plain = RoPEScheme(head_dim).build_inv_freq()
rotations = TRAIN_LEN / (2 * math.pi / plain)
ramp = ((rotations - 1.0) / (32.0 - 1.0)).clamp(0, 1)

print("YaRN per-band decision (target 512):")
print(f"{'band':>6} {'rotations':>11} {'ramp':>7}  treatment")
print("-" * 48)
for i in [0, 2, 4, 6, 8, 12, 15]:
    r = ramp[i].item()
    treat = ("keep original" if r > 0.95
             else "full interpolation" if r < 0.05
             else "ramped blend")
    print(f"{i:>6} {rotations[i]:>11.2f} {r:>7.2f}  {treat}")

print(f"\nAttention temperature scale: {yarn.attention_scale:.4f} "
      f"(sharpens the distribution)")

YaRN per-band decision (target 512):
  band   rotations    ramp  treatment
------------------------------------------------
     0       20.37    0.62  ramped blend
     2        6.44    0.18  ramped blend
     4        2.04    0.03  full interpolation
     6        0.64    0.00  full interpolation
     8        0.20    0.00  full interpolation
    12        0.02    0.00  full interpolation
    15        0.00    0.00  full interpolation

Attention temperature scale: 0.8782 (sharpens the distribution)


In [12]:
print(f"{'context':>9} {'base':>9} {'PI':>9} {'NTK':>9} {'YaRN':>9}")
print("-" * 48)
yarn_ppl = {}
for L in eval_lengths:
    scheme = YaRN(head_dim, train_len=TRAIN_LEN, target_len=L)
    yarn_ppl[L] = perplexity(model, scheme, L)
    print(f"{L:>9} {baseline_ppl[L]:>9.2f} {pi_ppl[L]:>9.2f} "
          f"{ntk_ppl[L]:>9.2f} {yarn_ppl[L]:>9.2f}")

  context      base        PI       NTK      YaRN
------------------------------------------------
      128     94.95     83.97     90.84     97.94
      192     99.55    203.98    113.98    101.15


      256    106.87    385.98    114.01    168.87
      384    131.06    537.12    139.36    221.07


      512    147.44    618.97    136.33    276.37


### 3.4 ALiBi — don't rotate, just penalize distance

A different philosophy. Drop positional embeddings entirely and add a **linear penalty**
to attention scores based on distance:

`score(i, j) = q_i · k_j / √d − m_h · (i − j)`

Each head gets its own slope `m_h` from a geometric sequence, so some heads look locally
(steep slope) and others globally (shallow slope). Nothing in this has a maximum position,
so it extrapolates by construction — the paper's title is literally *Train Short, Test Long*.

The trade-off: ALiBi's bias monotonically prefers recent tokens, so it is weaker at tasks
needing precise retrieval from far back. Most frontier models chose RoPE-plus-scaling
instead. ALiBi is architectural (you must train with it), so we train a second small model.

In [13]:
class ALiBi:
    """
    Attention with Linear Biases (Press et al. 2021).

    No rotation at all -- `tables()` returns None -- just a per-head distance
    penalty added to the scores.
    """

    name = "ALiBi"
    attention_scale = 1.0

    def __init__(self, num_heads):
        self.num_heads = num_heads
        # Geometric slopes: head 0 is the most global, the last the most local
        self.slopes = torch.tensor(
            [2 ** (-8 * (i + 1) / num_heads) for i in range(num_heads)]
        )

    def tables(self, seq_len):
        return None

    def bias(self, seq_len):
        """Returns (1, num_heads, seq_len, seq_len)."""
        positions = torch.arange(seq_len)
        # distance[i, j] = i - j, non-negative in the causal region
        distance = positions[:, None] - positions[None, :]
        return -self.slopes[None, :, None, None] * distance[None, None, :, :].float()


class NoPE:
    """
    No position information whatsoever.

    Included because the result is counter-intuitive: a *causal* decoder still
    has implicit position information. Token t can attend to t+1 keys while
    token 0 can attend to 1, so the number of visible tokens itself encodes
    position, and the model can learn to read it.
    """

    name = "NoPE"
    attention_scale = 1.0

    def tables(self, seq_len):
        return None

    def bias(self, seq_len):
        return None


print("ALiBi slopes per head:", [f"{s:.4f}" for s in ALiBi(4).slopes])

ALiBi slopes per head: ['0.2500', '0.0625', '0.0156', '0.0039']


In [14]:
def train_variant(scheme_factory, steps=700, label=""):
    """Train a fresh ToyLM with a fixed position scheme."""
    torch.manual_seed(0)
    m = ToyLM(tokenizer.vocab_size).to(device)
    scheme = scheme_factory(m)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=3e-3, total_steps=steps, pct_start=0.1
    )
    m.train()
    for _ in range(steps):
        x, y = get_batch(train_data, 24, TRAIN_LEN)
        _, loss = m(x, scheme, targets=y)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        sched.step()
    m.eval()
    print(f"  {label}: final train loss {loss.item():.4f}")
    return m, scheme


start = time.time()
print("Training architectural variants (each sees only 128-token windows):")
alibi_model, alibi_scheme = train_variant(
    lambda m: ALiBi(4), label="ALiBi"
)
nope_model, nope_scheme = train_variant(
    lambda m: NoPE(), label="NoPE "
)
print(f"Done in {time.time()-start:.1f}s")

Training architectural variants (each sees only 128-token windows):


  ALiBi: final train loss 0.0714


  NoPE : final train loss 0.3389
Done in 127.9s


In [15]:
print(f"{'context':>9} {'RoPE':>9} {'YaRN':>9} {'ALiBi':>9} {'NoPE':>9}")
print("-" * 48)
alibi_ppl, nope_ppl = {}, {}
for L in eval_lengths:
    alibi_ppl[L] = perplexity(alibi_model, alibi_scheme, L)
    nope_ppl[L] = perplexity(nope_model, nope_scheme, L)
    print(f"{L:>9} {baseline_ppl[L]:>9.2f} {yarn_ppl[L]:>9.2f} "
          f"{alibi_ppl[L]:>9.2f} {nope_ppl[L]:>9.2f}")

  context      RoPE      YaRN     ALiBi      NoPE
------------------------------------------------
      128     94.95     97.94     98.75     68.08


      192     99.55    101.15     98.95     71.28


      256    106.87    168.87    104.83     89.61


      384    131.06    221.07    107.78     94.40


      512    147.44    276.37    107.97    113.16


### The comparison plot

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

series = [
    ('RoPE (no scaling)', baseline_ppl, '#C73E1D', 'o-'),
    ('Position Interpolation', pi_ppl, '#F18F01', 's-'),
    ('NTK-aware', ntk_ppl, '#2E86AB', '^-'),
    ('YaRN', yarn_ppl, '#3B7A57', 'd-'),
    ('ALiBi', alibi_ppl, '#A23B72', 'v-'),
    ('NoPE', nope_ppl, 'gray', 'x--'),
]

for label, data, color, style in series:
    axes[0].plot(eval_lengths, [data[L] for L in eval_lengths], style,
                 color=color, label=label)
axes[0].axvline(TRAIN_LEN, color='black', ls=':', lw=1.5, label='trained length')
axes[0].set_xlabel('evaluation context length')
axes[0].set_ylabel('perplexity (lower is better)')
axes[0].set_yscale('log')
axes[0].set_title('Extrapolation beyond the training length')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Relative degradation from each method's own baseline at TRAIN_LEN
for label, data, color, style in series:
    rel = [data[L] / data[TRAIN_LEN] for L in eval_lengths]
    axes[1].plot(eval_lengths, rel, style, color=color, label=label)
axes[1].axhline(1.0, color='black', ls=':', lw=1.5)
axes[1].set_xlabel('evaluation context length')
axes[1].set_ylabel('perplexity relative to its own 128-token score')
axes[1].set_title('Degradation factor (1.0 = no loss from longer context)')
axes[1].set_yscale('log')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Unscaled RoPE is the outlier. Every scaling method keeps degradation")
print("bounded, and the per-band methods (NTK, YaRN) do better than uniform PI")
print("because they leave the well-trained high frequencies alone.")
print("\nCaveat worth stating: this is a 0.6M-parameter character model on 11KB")
print("of Shakespeare. The ORDERING reproduces the literature; the absolute")
print("numbers do not transfer. At real scale YaRN reliably beats NTK beats PI,")
print("and all three want a few hundred fine-tuning steps at the new length.")

Unscaled RoPE is the outlier. Every scaling method keeps degradation
bounded, and the per-band methods (NTK, YaRN) do better than uniform PI
because they leave the well-trained high frequencies alone.

Caveat worth stating: this is a 0.6M-parameter character model on 11KB
of Shakespeare. The ORDERING reproduces the literature; the absolute
numbers do not transfer. At real scale YaRN reliably beats NTK beats PI,
and all three want a few hundred fine-tuning steps at the new length.


## 4. Sliding windows and the attention sink

A completely different angle on long context: **don't attend to everything.** Restrict each
token to the last `W` keys. Cost becomes linear in sequence length, and the KV cache stops
growing. Mistral-7B shipped with `W = 4096`.

The naive version has a spectacular failure mode, discovered by the StreamingLLM authors:
as soon as the window slides past the **first few tokens**, quality collapses.

The explanation is that softmax must sum to 1. When a head has nothing it wants to attend
to, it needs somewhere to dump its probability mass — and models learn to use the first
few positions as that dumping ground. These are **attention sinks**: positions whose
content barely matters but whose *availability* does. Evict them and every head is forced
to distribute mass over tokens it actively does not want, corrupting the output.

The fix is almost trivially cheap: keep the first ~4 tokens permanently in the cache.

In [17]:
WINDOW = 64   # half the training length, so the window genuinely slides

print(f"Sliding window W={WINDOW}, evaluated at increasing lengths")
print(f"{'context':>9} {'full attn':>11} {'window only':>13} {'window+4 sinks':>16}")
print("-" * 54)

window_ppl, sink_ppl = {}, {}
for L in eval_lengths:
    scheme = YaRN(head_dim, train_len=TRAIN_LEN, target_len=L)
    window_ppl[L] = perplexity(model, scheme, L, window=WINDOW, num_sink=0)
    sink_ppl[L] = perplexity(model, scheme, L, window=WINDOW, num_sink=4)
    print(f"{L:>9} {yarn_ppl[L]:>11.2f} {window_ppl[L]:>13.2f} "
          f"{sink_ppl[L]:>16.2f}")

improvement = [window_ppl[L] / sink_ppl[L] for L in eval_lengths]
print(f"\nKeeping 4 extra tokens improves perplexity by "
      f"{sum(improvement)/len(improvement):.2f}x on average.")
print(f"Cache cost of that fix: 4 positions out of {WINDOW}.")

Sliding window W=64, evaluated at increasing lengths
  context   full attn   window only   window+4 sinks
------------------------------------------------------


      128       97.94         91.29            83.36


      192      101.15        132.98           126.75


      256      168.87        160.98           153.18


      384      221.07        207.00           207.92


      512      276.37        249.12           249.20

Keeping 4 extra tokens improves perplexity by 1.04x on average.
Cache cost of that fix: 4 positions out of 64.


### Seeing the sink directly

The claim is that models dump attention on early positions. We can just look.

In [18]:
@torch.no_grad()
def attention_to_positions(model, scheme, seq_len=256):
    """Average attention received by each key position, across heads and layers."""
    x, _ = get_batch(val_data, 2, seq_len)
    h = model.embed(x)
    received = torch.zeros(seq_len)
    n = 0
    for block in model.blocks:
        normed = block.attn_norm(h)
        attn = block.attn
        B, T, d = normed.shape
        qkv = attn.W_qkv(normed).view(B, T, 3, attn.num_heads, attn.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        cos_sin = scheme.tables(T)
        if cos_sin is not None:
            q = apply_rotation(q, *cos_sin)
            k = apply_rotation(k, *cos_sin)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
        scores = scores.masked_fill(
            ~torch.ones(T, T, dtype=torch.bool).tril(), float('-inf')
        )
        w = F.softmax(scores, dim=-1)
        received += w.mean(dim=(0, 1)).sum(dim=0)   # total mass each key gets
        n += 1
        h = h + attn(block.attn_norm(h), scheme)
        h = h + block.ffn(block.ffn_norm(h))
    return received / n


scheme = YaRN(head_dim, train_len=TRAIN_LEN, target_len=256)
received = attention_to_positions(model, scheme, 256)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(received.numpy(), color='#2E86AB', lw=1)
axes[0].set_xlabel('key position')
axes[0].set_ylabel('total attention mass received')
axes[0].set_title('Early positions absorb disproportionate attention')
axes[0].grid(alpha=0.3)

axes[1].bar(range(16), received[:16].numpy(), color='#C73E1D')
axes[1].set_xlabel('key position (first 16)')
axes[1].set_ylabel('total attention mass received')
axes[1].set_title('Zoomed: the sink')
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

first4 = received[:4].sum().item()
total = received.sum().item()
print(f"Positions 0-3 absorb {100*first4/total:.1f}% of all attention mass,")
print(f"despite being {100*4/256:.1f}% of the positions.")
print("\nThat concentration is why evicting them is so damaging -- and why")
print("some architectures now add a *dedicated* learned sink token, giving")
print("heads an explicit no-op target instead of hijacking real content.")

Positions 0-3 absorb 4.0% of all attention mass,
despite being 1.6% of the positions.

That concentration is why evicting them is so damaging -- and why
some architectures now add a *dedicated* learned sink token, giving
heads an explicit no-op target instead of hijacking real content.


## 5. The real constraint: KV cache memory

Everything above was about whether the *math* survives long sequences. In production the
binding constraint is usually memory. Notebook 09 introduced the KV cache; at long context
it becomes the dominant consumer, and it scales with **batch × length**, not with model
size.

Let's compute it for real configurations.

In [19]:
def kv_cache_gb(num_layers, num_kv_heads, head_dim, seq_len, batch_size=1,
                bytes_per_element=2):
    """KV cache size in GB. Factor of 2 for storing both K and V."""
    elements = (
        2 * num_layers * num_kv_heads * head_dim * seq_len * batch_size
    )
    return elements * bytes_per_element / 1e9


configs = [
    # name,                 layers, q_heads, kv_heads, head_dim, weights_gb
    ("Llama-2-7B (MHA)",        32,      32,       32,       128,  13.5),
    ("Llama-3-8B (GQA 4:1)",    32,      32,        8,       128,  16.0),
    ("Llama-3-70B (GQA 8:1)",   80,      64,        8,       128, 141.0),
]

print("KV cache at batch size 1, fp16:\n")
print(f"{'model':<24} {'weights':>9} {'8k':>9} {'32k':>9} {'128k':>9} {'1M':>9}")
print("-" * 74)
for name, layers, q_heads, kv_heads, hd, weights in configs:
    sizes = [
        kv_cache_gb(layers, kv_heads, hd, L)
        for L in (8192, 32768, 131072, 1048576)
    ]
    print(f"{name:<24} {weights:>8.1f}G "
          + " ".join(f"{s:>8.1f}G" for s in sizes))

print("\nAt 128k context Llama-3-8B's cache is comparable to its weights.")
print("At 1M it is several times larger. And that is batch size ONE -- the")
print("cache scales linearly with concurrent requests while the weights are")
print("shared, so at any serving batch size the cache dominates completely.")

KV cache at batch size 1, fp16:

model                      weights        8k       32k      128k        1M
--------------------------------------------------------------------------
Llama-2-7B (MHA)             13.5G      4.3G     17.2G     68.7G    549.8G
Llama-3-8B (GQA 4:1)         16.0G      1.1G      4.3G     17.2G    137.4G
Llama-3-70B (GQA 8:1)       141.0G      2.7G     10.7G     42.9G    343.6G

At 128k context Llama-3-8B's cache is comparable to its weights.
At 1M it is several times larger. And that is batch size ONE -- the
cache scales linearly with concurrent requests while the weights are
shared, so at any serving batch size the cache dominates completely.


In [20]:
# What GQA already bought us, and what MHA would have cost
mha_128k = kv_cache_gb(32, 32, 128, 131072)
gqa_128k = kv_cache_gb(32, 8, 128, 131072)
print(f"Llama-3-8B at 128k with MHA (32 kv heads): {mha_128k:.1f} GB")
print(f"                    with GQA (8 kv heads): {gqa_128k:.1f} GB")
print(f"GQA alone is a {mha_128k/gqa_128k:.0f}x reduction -- which is why every")
print("long-context model uses it. Notebook 16 covers MLA, which goes further.")

Llama-3-8B at 128k with MHA (32 kv heads): 68.7 GB
                    with GQA (8 kv heads): 17.2 GB
GQA alone is a 4x reduction -- which is why every
long-context model uses it. Notebook 16 covers MLA, which goes further.


### Three ways to shrink the cache

**Quantize it.** The cache is activations, and activations tolerate low precision better
than weights do. fp16 → int8 halves it; int4 quarters it. This is the cheapest win
available and is standard in vLLM and TensorRT-LLM.

**Evict from it.** Not every token needs keeping. H2O and SnapKV observe that attention is
sparse and concentrated on a stable subset of "heavy hitter" tokens, so the rest can be
dropped with modest quality loss. The risk is task-dependence: a token that looks
unimportant now may be exactly what a later question asks about.

**Share it across layers.** Cross-Layer Attention (CLA) and YOCO have multiple layers reuse
one set of keys and values, cutting the cache by the sharing factor. This is GQA's idea
applied along depth instead of across heads.

Let's quantify the first, since it is exact and easy to measure.

In [21]:
def quantize_per_token(tensor, bits=8):
    """
    Symmetric per-token quantization -- the granularity real KV quantizers use.

    Each token's key/value vector gets its own scale, which matters because
    activation magnitude varies a lot between tokens.
    """
    qmax = 2 ** (bits - 1) - 1
    scale = tensor.abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / qmax
    q = (tensor / scale).round().clamp(-qmax - 1, qmax)
    return q * scale


@torch.no_grad()
def perplexity_with_quantized_cache(model, scheme, seq_len, bits, num_batches=8):
    """
    Perplexity when keys and values are quantized.

    Quantizing K and V inside attention is equivalent to quantizing the cache,
    since the cache holds exactly those tensors.
    """
    total = 0.0
    for _ in range(num_batches):
        x, y = get_batch(val_data, 4, seq_len)
        h = model.embed(x)
        for block in model.blocks:
            normed = block.attn_norm(h)
            attn = block.attn
            B, T, d = normed.shape
            qkv = attn.W_qkv(normed).view(B, T, 3, attn.num_heads, attn.head_dim)
            q, k, v = qkv.permute(2, 0, 3, 1, 4)
            cos_sin = scheme.tables(T)
            if cos_sin is not None:
                q = apply_rotation(q, *cos_sin)
                k = apply_rotation(k, *cos_sin)
            if bits is not None:
                k = quantize_per_token(k, bits)
                v = quantize_per_token(v, bits)
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(attn.head_dim)
            scores = scores.masked_fill(
                ~torch.ones(T, T, dtype=torch.bool).tril(), float('-inf')
            )
            out = torch.matmul(F.softmax(scores, dim=-1), v)
            out = out.transpose(1, 2).contiguous().view(B, T, d)
            h = h + attn.W_o(out)
            h = h + block.ffn(block.ffn_norm(h))
        logits = model.head(model.norm(h))
        total += F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), y.reshape(-1)
        ).item()
    return math.exp(total / num_batches)


L = 256
scheme = YaRN(head_dim, train_len=TRAIN_LEN, target_len=L)

print(f"KV cache quantization at context {L}:\n")
print(f"{'precision':>12} {'perplexity':>12} {'vs fp32':>10} {'cache size':>12}")
print("-" * 50)
ref = perplexity_with_quantized_cache(model, scheme, L, bits=None)
print(f"{'fp32':>12} {ref:>12.3f} {'1.000':>10} {'100%':>12}")
for bits in (8, 4, 3, 2):
    ppl = perplexity_with_quantized_cache(model, scheme, L, bits=bits)
    print(f"{f'int{bits}':>12} {ppl:>12.3f} {ppl/ref:>10.3f} "
          f"{f'{100*bits/32:.0f}%':>12}")

print("\nint8 is essentially free. int4 costs a little. Below that it degrades")
print("quickly. That matches production practice: int8 KV cache is a default,")
print("int4 is a considered trade, int2 is research.")

KV cache quantization at context 256:

   precision   perplexity    vs fp32   cache size
--------------------------------------------------


        fp32      168.584      1.000         100%


        int8      158.837      0.942          25%


        int4      167.951      0.996          12%


        int3      180.606      1.071           9%


        int2      278.023      1.649           6%

int8 is essentially free. int4 costs a little. Below that it degrades
quickly. That matches production practice: int8 KV cache is a default,
int4 is a considered trade, int2 is research.


## 6. Measuring long context honestly

Perplexity is a **weak** proxy for long-context ability. A model can score well on
perplexity at 128k because most next-token predictions are locally determined — it never
had to use the far context at all.

The standard probe is **needle in a haystack**: hide a specific fact in a long document and
ask for it. Let's build a character-level version. Our toy model has no instruction
following, so we measure the sharpest available signal: does the model assign higher
probability to the needle's continuation when the needle is present than when it is not?

In [22]:
@torch.no_grad()
def needle_signal(model, scheme, context_len, depth_fraction, window=None, num_sink=0):
    """
    Insert a distinctive string, then measure how well the model predicts its
    continuation from far away.

    Returns the mean log-probability of the needle's own tokens. If position
    encoding has broken, this collapses regardless of perplexity.
    """
    needle = "the secret code is zebra"
    needle_ids = tokenizer.encode(needle)

    filler = val_data[:context_len].clone()
    if len(filler) < context_len:
        reps = context_len // len(val_data) + 1
        filler = val_data.repeat(reps)[:context_len]

    insert_at = int((context_len - len(needle_ids) - 1) * depth_fraction)
    seq = filler.clone()
    seq[insert_at:insert_at + len(needle_ids)] = torch.tensor(needle_ids)

    x = seq[:-1].unsqueeze(0)
    y = seq[1:].unsqueeze(0)
    logits, _ = model(x, scheme, window=window, num_sink=num_sink)
    logprobs = F.log_softmax(logits, dim=-1)

    # Score only the needle's own tokens
    lo = insert_at
    hi = insert_at + len(needle_ids) - 1
    picked = logprobs[0, lo:hi].gather(-1, y[0, lo:hi].unsqueeze(-1))
    return picked.mean().item()


print("Mean log-prob of the needle tokens (higher is better, 0 is perfect):\n")
print(f"{'context':>9} {'RoPE':>9} {'YaRN':>9} {'ALiBi':>9} {'win only':>10} {'win+sink':>10}")
print("-" * 62)
for L in [128, 256, 384, 512]:
    plain_s = RoPEScheme(head_dim)
    yarn_s = YaRN(head_dim, train_len=TRAIN_LEN, target_len=L)
    row = [
        needle_signal(model, plain_s, L, 0.3),
        needle_signal(model, yarn_s, L, 0.3),
        needle_signal(alibi_model, alibi_scheme, L, 0.3),
        needle_signal(model, yarn_s, L, 0.3, window=WINDOW, num_sink=0),
        needle_signal(model, yarn_s, L, 0.3, window=WINDOW, num_sink=4),
    ]
    print(f"{L:>9} " + " ".join(f"{v:>9.3f}" for v in row[:3])
          + " " + " ".join(f"{v:>10.3f}" for v in row[3:]))

print("\nUnscaled RoPE degrades fastest. Note also that a sliding window cannot")
print("retrieve a needle older than the window, no matter how many sinks you")
print("keep -- linear attention cost is paid for in retrieval range. That is")
print("the trade hybrid architectures try to escape (notebook 16).")

Mean log-prob of the needle tokens (higher is better, 0 is perfect):

  context      RoPE      YaRN     ALiBi   win only   win+sink
--------------------------------------------------------------
      128    -5.332    -5.332    -5.375     -5.332     -5.332
      256    -4.865    -5.888    -5.483     -5.982     -5.971


      384    -4.122    -5.498    -5.203     -5.239     -5.238
      512    -4.378    -5.485    -5.344     -5.742     -5.727

Unscaled RoPE degrades fastest. Note also that a sliding window cannot
retrieve a needle older than the window, no matter how many sinks you
keep -- linear attention cost is paid for in retrieval range. That is
the trade hybrid architectures try to escape (notebook 16).


### What real long-context evaluation looks like

| Benchmark | What it tests | Why it matters |
|---|---|---|
| **Needle in a Haystack** | Retrieve one fact from a long document | The floor. Failing it means the context is decorative. |
| **RULER** | Multi-needle, tracing, aggregation at controlled lengths | Reveals that "effective" context is often 2–8x shorter than advertised |
| **LongBench / ∞Bench** | Realistic tasks: QA, summarization, code | Closer to use, harder to interpret |
| **Perplexity at length** | Average next-token loss | Cheap, and the weakest signal — passes even when retrieval fails |

The consistent finding from RULER is worth internalizing: **advertised context length and
usable context length are different numbers.** A model claiming 128k may degrade
substantially past 32k on tasks that genuinely require the whole window. Always measure on
your own task rather than trusting the specification.

## 7. Long context by parallelism

One more approach, for completeness. Everything above tries to make a *single device*
handle a long sequence. The alternative is to split the sequence across devices.

**Ring Attention** shards the sequence across GPUs, with each device holding a block of
keys and values. Blocks rotate around a logical ring so that every query eventually meets
every key, with communication overlapped behind computation. Memory per device becomes
`O(N/P)`, so context scales with the number of devices — a genuinely different scaling axis
from anything in this notebook.

This is *sequence parallelism*, and it belongs with the other parallelism strategies.
Notebook 22 covers it alongside data, tensor, pipeline, and expert parallelism.

## Summary

We broke a model by feeding it longer sequences than it was trained on, then fixed it four
different ways.

**The failure.** RoPE's rotation angle is `pos · inv_freq`, and `pos` is unbounded. Past the
training length the low-frequency bands reach angles never observed, and perplexity
explodes — in our run, several times worse with 4x more context. *More information made the
model worse.*

**The diagnosis.** Frequency bands are not equal. High-frequency bands complete many
rotations inside the training window and extrapolate fine; low-frequency bands never
complete one, and they are what breaks. Every good fix follows from that observation.

**The fixes, in order of sophistication**

| Method | Mechanism | Cost |
|---|---|---|
| Position Interpolation | Divide all positions by `s` | Uniform — degrades local resolution too |
| NTK-aware | Raise the base, which slows slow bands more | Nearly free; implicit per-band targeting |
| YaRN | Explicit per-band ramp + attention temperature | Best quality; more knobs |
| ALiBi | No rotation, linear distance penalty | Extrapolates by construction; weaker at long retrieval |

**Sliding window + attention sink.** Restricting attention to the last `W` keys makes cost
linear, but evicting the first few tokens collapses quality — those positions absorb a
disproportionate share of attention mass (over a third from four positions in our
measurement) because softmax needs somewhere to put unwanted probability. Keeping ~4 tokens
pinned fixes it almost for free.

**The real limit is memory.** At 128k context, Llama-3-8B's KV cache rivals its weights,
and it scales with batch size while weights are shared. GQA is already a 4x reduction;
int8 quantization of the cache is nearly lossless and halves it again.

### Key Takeaways

1. **Advertised context ≠ usable context.** Measure on your task. RULER-style evaluation
   routinely finds effective context 2–8x below the specification.
2. **Position extrapolation fails in the low-frequency bands.** That single fact explains
   PI, NTK-aware scaling, and YaRN.
3. **Don't scale what isn't broken.** PI compresses every band; NTK and YaRN spare the
   high frequencies and do better as a result.
4. **YaRN's attention temperature is not optional.** Interpolation flattens attention;
   the `1/(0.1·ln s + 1)` correction sharpens it back.
5. **Attention sinks are real and cheap to respect.** Softmax must sum to 1, so heads need
   a dumping ground. Never evict position 0.
6. **KV cache is the production constraint, not perplexity.** It scales with
   `batch × length × layers × kv_heads` — which is exactly what GQA, MLA, quantization,
   eviction, and cross-layer sharing all attack.
7. **Perplexity is the weakest long-context metric.** It can look fine while retrieval is
   entirely broken.

### Self-check

- Why does perplexity *rise* past the training length, when more context should help?
- Which RoPE frequency bands break first, and why?
- What exactly does NTK-aware scaling do differently from Position Interpolation, and why
  is it better?
- In YaRN, what does `r_i = train_len / λ_i` measure, and what decision does it drive?
- Why does YaRN need an attention temperature correction at all?
- Why does evicting the first four tokens of a sliding-window cache hurt so much?
- A 70B model serving 32 concurrent requests at 32k context: is the KV cache bigger or
  smaller than the weights? Compute it.
- Your model passes needle-in-a-haystack at 128k. What has that *not* told you?

### What's next

Notebook 16 attacks the quadratic cost itself: **MLA**, linear attention, state-space
models (Mamba), and why hybrid architectures are winning.

### References

- Su et al., 2021 — [RoFormer / RoPE](https://arxiv.org/abs/2104.09864)
- Chen et al., 2023 — [Extending Context Window via Position Interpolation](https://arxiv.org/abs/2306.15595)
- bloc97, 2023 — [NTK-Aware Scaled RoPE](https://www.reddit.com/r/LocalLLaMA/comments/14lz7j5/ntkaware_scaled_rope_allows_llama_models_to_have/)
- Peng et al., 2023 — [YaRN: Efficient Context Window Extension](https://arxiv.org/abs/2309.00071)
- Press et al., 2021 — [Train Short, Test Long: ALiBi](https://arxiv.org/abs/2108.12409)
- Kazemnejad et al., 2023 — [The Impact of Positional Encoding on Length Generalization](https://arxiv.org/abs/2305.19466) (NoPE)
- Xiao et al., 2023 — [Efficient Streaming LMs with Attention Sinks](https://arxiv.org/abs/2309.17453)
- Jiang et al., 2023 — [Mistral 7B](https://arxiv.org/abs/2310.06825) (sliding-window attention)
- Zhang et al., 2023 — [H2O: Heavy-Hitter Oracle for KV Cache](https://arxiv.org/abs/2306.14048)
- Li et al., 2024 — [SnapKV](https://arxiv.org/abs/2404.14469)
- Brandon et al., 2024 — [Reducing Transformer KV Cache with Cross-Layer Attention](https://arxiv.org/abs/2405.12981)
- Hsieh et al., 2024 — [RULER: What's the Real Context Size of Your Long-Context LMs?](https://arxiv.org/abs/2404.06654)
- Liu et al., 2023 — [Ring Attention with Blockwise Transformers](https://arxiv.org/abs/2310.01889)
- Liu et al., 2023 — [Lost in the Middle](https://arxiv.org/abs/2307.03172)